# 04 · HCHO Hotspot Detection & Spatial Attribution

Detecting localized formaldehyde enhancements via:
1. **PHV** (Percentage Higher than Vicinity, Dong et al. 2026)
2. **Getis-Ord Gi\*** spatial statistics
3. **Isolation Forest** anomaly baseline


In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

from isro_aqi.synthetic import SyntheticConfig, generate_all
from isro_aqi.hcho.phv import detect_hotspots, phv_field
from isro_aqi.hcho.getis_ord import getis_ord_gi_star


### 1. Compute PHV and Gi\* on HCHO Field


In [ ]:
cfg = SyntheticConfig(resolution_deg=0.5, n_days=15)
data = generate_all(cfg)
hcho_slice = data["stack"]["hcho"].isel(time=5)

phv_map = phv_field(hcho_slice.values)
z_scores, p_vals = getis_ord_gi_star(hcho_slice.values, radius=2)

print(f"PHV max: {np.nanmax(phv_map):.2f}, mean: {np.nanmean(phv_map):.2f}")
print(f"Gi* z-score max: {np.nanmax(z_scores):.2f}")


### 2. Visualize Detected Anomaly Zones


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

hcho_slice.plot(ax=axes[0], cmap="magma")
axes[0].set_title("Original HCHO Column")

im1 = axes[1].imshow(phv_map, origin="lower", cmap="viridis", vmin=0.8, vmax=1.8)
axes[1].set_title("PHV Field (Local Anomaly Ratio)")
plt.colorbar(im1, ax=axes[1])

im2 = axes[2].imshow(z_scores, origin="lower", cmap="coolwarm", vmin=-3, vmax=3)
axes[2].set_title("Getis-Ord Gi* z-scores (Hotspots)")
plt.colorbar(im2, ax=axes[2])

plt.tight_layout()
plt.show()
